In [1]:
import re

class UltimateISRIStemmer:
    def __init__(self):
        # السوابق واللواحق مرتبة من الأطول للأقصر
        self.prefixes = ["وال", "بال", "كال", "فال", "ال", "لل", "و", "ف", "ب"]
        self.suffixes = ["ون", "ين", "ان", "ات", "وا", "تم", "كن", "نا", "كم", "هم", "ها", "ت", "ي", "ه"]

    def clean(self, word):
        # إزالة التشكيل والتطويل وتوحيد الهمزات
        word = re.sub(r'[ًٌٍَُِّْـ]', '', word)
        word = re.sub(r'[أإآ]', 'ا', word)
        return word

    def stem(self, word):
        word = self.clean(word)
        
        # 1. إزالة السوابق
        for p in self.prefixes:
            if word.startswith(p) and len(word) > 3:
                word = word[len(p):]
                break
        
        # 2. إزالة اللواحق
        for s in self.suffixes:
            if word.endswith(s) and len(word) > 3:
                word = word[:-len(s)]
                break

        # 3. معالجة حروف الزيادة الخاصة (السين، التاء، الميم، الألف)
        # حالة: استخراج -> خرج
        if word.startswith("است") and len(word) >= 6:
            word = word[3:]
        
        # حالة: المسافرون (بعد الحذف تصبح مسافر) -> سفر
        if len(word) >= 4:
            # إذا بدأت بميم زائدة (مفاعل، مفعول، مفعل)
            if word.startswith('م'): word = word[1:]
            # إذا كان الحرف الثاني ألف زائدة (فاعل)
            if len(word) >= 3 and word[1] == 'ا':
                word = word[0] + word[2:]
            # إذا بدأت بياء أو تاء المضارعة (يتحدث، تتحدث)
            if word.startswith(('ي', 'ت')) and len(word) >= 4:
                word = word[1:]

        return word[:3] # ضمان العودة بجذر ثلاثي

# --- تجربة النتائج ---
isri = UltimateISRIStemmer()
words = ["المسافرون", "يتحدثون", "بالجامعات", "استخراج"]

print(f"{'الكلمة':<12} | {'الناتج المطور':<12} | {'الحالة':<10}")
print("-" * 40)
for w in words:
    root = isri.stem(w)
    status = "✅ صحيح" if root in ["سفر", "حدث", "جمع", "خرج"] else "❌ يحتاج تعديل"
    print(f"{w:<12} | {root:<12} | {status}")

الكلمة       | الناتج المطور | الحالة    
----------------------------------------
المسافرون    | سفر          | ✅ صحيح
يتحدثون      | تحد          | ❌ يحتاج تعديل
بالجامعات    | جمع          | ✅ صحيح
استخراج      | خرا          | ❌ يحتاج تعديل


In [12]:
# ما الذي تم تحسينه في هذا الكود؟
# 1 توحيد الحروف (Normalization): الكود الأصلي في ISRI غالباً لا يعالج الهمزات (أ، إ، آ) بشكل موحد، مما يشتت الخوارزمية. هنا قمنا بدمجها جميعاً في "ا".

# 2 استراتيجية الحذف الطويل (Greedy Match): أضفنا sorted(..., reverse=True) لضمان حذف "والـ" قبل "الـ" و "و"، وهذا يمنع بقاء حروف زائدة ملتصقة بالجذر.

# 3 حماية الجذور القصيرة: أضفنا شرط len(word) > 3 قبل أي عملية حذف، لضمان عدم ضياع حروف الكلمات الثنائية أو الثلاثية الأصلية (مثل "أب" أو "يد").

# 4 معالجة حروف المضارعة: أضفنا منطقاً ذكياً في "المرحلة 2" للتعامل مع حروف "أنيت" (حروف المضارعة) التي تأتي في بداية الأفعال، وهي ميزة غالباً ما تفتقدها التطبيقات البسيطة لـ ISRI.